# Build a configurable ReAct agent

You will build **one** web-research agent that runs two different ways. Pass `quick` and you get a
fast, shallow search on a cheap model with a concise persona. Pass `deep` and you get a wide
search on a stronger model with a thorough persona.

Same prompt. Same tool. Same code. The only thing that changes is **one string**.

That string is an **alias**. By the end you will have changed a running agent's model, its
personality and how hard it searches, without a redeploy, a code change, or a restart.

| Piece | What it does | Who runs it |
|---|---|---|
| `web_research` | the tool in the catalog: one `query` argument, and **no opinion about depth** | the platform stores it |
| `search_the_web` | the Python that calls Tavily, at whichever depth the alias asked for | **your code** |
| `web-research-agent` v1 | the quick persona, on the cheap model | the platform stores it |
| `web-research-agent` v2 | the deep persona, on the stronger model | the platform stores it |
| aliases `quick` and `deep` | named pointers: `quick` to v1, `deep` to v2 | the platform stores them |
| `run_prompt_with_tools` | the loop: model turn, tool call, model turn again | the SDK |

Every cell runs against a real account, two real gateway models, and the real Tavily search API.
Nothing here is faked or mocked.

**Reading aliases is REST-only today.** Both SDKs can *write* an alias — `promote_alias` — but
neither wraps `GET /prompts/<id>/aliases`, so the cells that show you where each alias points call
that endpoint with `httpx`. Do not go looking for `hub.prompts.list_aliases`; it does not exist.

**This notebook needs two models.** That is the point — the alias has to be able to change the
model, so there must be two to change between. The preflight checks for both and tells you what
to do if one is missing.

**Two ways to do every step.** Each step that creates something has two headings:
**In the dashboard**, with the values to type and a screenshot, and **The same thing in code**,
with a cell to run. They are not two different features — the dashboard and the SDK call the same
API, so the result is identical. Pick either. Doing both is harmless, because every code cell
looks for what already exists before it creates anything.

**Three kinds of code cell.** Most of this notebook is not the thing you would ship. Every cell's
lead-in says which kind it is:

| Label | What it is | Goes in your app? |
|---|---|---|
| **Setup** | creates something on the platform, once. The dashboard does the same job. | no |
| **Your app** | the code that would really ship | **yes** |
| **Check** | proves the step worked, or shows what just happened | no |
| **Broken on purpose** | a failure being demonstrated | no |

**Companion page:** [Build a configurable ReAct agent](https://docs.acruxcore.com/docs/tutorials/build-a-configurable-react-agent)

---

## Step 0 — What you need before you start

**1. A personal API key.** **Account & keys → New key**, named `configurable-react-agent`. Copy
it the moment it appears — that is the only time the full value is shown.

**2. A Tavily API key.** The free tier is enough. Get one at [tavily.com](https://tavily.com/).

**3. Two models connected to the gateway.** A cheap one and a stronger one. This notebook uses
`gemini-flash` and `claude-haiku`, but any two will do — they only have to be *different*. Add
them under **Gateway → Models → New model**, one row each:

| Field | Cheap model | Stronger model |
|---|---|---|
| **Public name** | `gemini-flash` | `claude-haiku` |
| **Credential** | any credential that serves it | any credential that serves it |
| **Upstream model** | the provider's own id, for example `google/gemini-3.7-flash` | the provider's own id, for example `claude-haiku` |

If your two are named something else, change `QUICK_MODEL` and `DEEP_MODEL` in the next cell.
Nothing else in the notebook needs editing.

**4. `acruxcore` and `requests`.**

In [ ]:
%pip install -q --upgrade acruxcore requests

**Setup.** Set the keys and name everything this notebook will create.

A key typed into a notebook is saved *inside the notebook file*. Prefer setting these in your
shell before you start Jupyter, and treat this cell as a fallback.

In [1]:
import json
import os

# Better: export these in your shell before starting Jupyter.
os.environ.setdefault("ACRUXCORE_API_KEY", "acx_sk_...")
os.environ.setdefault("ACRUXCORE_BASE_URL", "https://api.acruxcore.com/api/v1")
os.environ.setdefault("TAVILY_API_KEY", "tvly-...")

TAVILY_KEY = os.environ["TAVILY_API_KEY"]

QUICK_MODEL = "gemini-flash"     # the cheap, fast one -> alias `quick` -> v1
DEEP_MODEL = "claude-haiku"      # the stronger one    -> alias `deep`  -> v2

PROMPT = "web-research-agent"    # the prompt this notebook creates
TOOL = "web_research"            # the tool this notebook creates

# Do NOT print the base URL: the saved output would publish whatever host you ran against.

### Preflight

**Check.** Four things can be wrong before anything interesting happens, and they fail in this
order: your AcruxCore key, the two models, your Tavily key, and whether this SDK version knows
`client_tools`. Checking them separately means one clear line instead of a stack trace out of the
middle of the loop.

In [2]:
import inspect

import httpx
import requests

import acruxcore
from acruxcore import AcruxCore
from acruxcore.gateway_api import GatewayNamespace

hub = AcruxCore()          # reads ACRUXCORE_API_KEY / ACRUXCORE_BASE_URL

# 1. Does the AcruxCore key work?
await hub.prompts.list(limit=1)
print("acruxcore key: ok")

# One raw HTTP client for the endpoints no SDK namespace covers. Closed in Step 13.
rest = httpx.AsyncClient(
    base_url=os.environ["ACRUXCORE_BASE_URL"],
    headers={"Authorization": f"Bearer {os.environ['ACRUXCORE_API_KEY']}"},
    timeout=60,
)


async def list_prompt_aliases(prompt_id: str) -> dict:
    """Every alias on a prompt, as {alias: version_number}.

    A notebook helper, NOT an SDK function. Both SDKs can promote an alias but neither
    can list them, so this calls GET /prompts/<id>/aliases directly.
    """
    res = await rest.get(f"/prompts/{prompt_id}/aliases")
    res.raise_for_status()
    return {row["alias"]: row["versionNumber"] for row in res.json()}


# 2. Are BOTH models connected? Models have no SDK namespace either.
res = await rest.get("/gateway/models")
available = [m["publicName"] for m in res.json()]
print("models on this team:", available or "NONE - add two in Gateway -> Models")
for label, model in (("QUICK_MODEL", QUICK_MODEL), ("DEEP_MODEL", DEEP_MODEL)):
    print(f"  {label} {model!r}:", "ok" if model in available else "MISSING")
if QUICK_MODEL == DEEP_MODEL:
    print("  !! both names are the same - the alias would change the persona but not the model")

# 3. Does the Tavily key work? One cheap search.
probe = requests.post("https://api.tavily.com/search",
                      headers={"Authorization": f"Bearer {TAVILY_KEY}"},
                      json={"query": "test", "max_results": 1}, timeout=45)
print("tavily key:", "ok" if probe.ok else f"FAILED {probe.status_code}")

# 4. Does this SDK version know client_tools? Step 9 needs it.
has_client_tools = "client_tools" in inspect.signature(
    GatewayNamespace.run_prompt_with_tools).parameters
print("client_tools supported:", has_client_tools)
if not has_client_tools:
    print(f"  !! acruxcore {acruxcore.__version__} is too old - upgrade it")

acruxcore key: ok
models on this team: ['mistral-small', 'llama-3.3-70b', 'claude-haiku', 'gemini-flash', 'gpt-4o-mini']
  QUICK_MODEL 'gemini-flash': ok
  DEEP_MODEL 'claude-haiku': ok
tavily key: ok
client_tools supported: True


---

## Step 1 — What an alias actually is

### The general problem

Every application needs settings that change without a rebuild. Which model, which wording, how
hard to search. The usual answers are an environment variable, a feature flag, or a config file —
and all three share one weakness: the setting and the thing it configures live in different
places, so they drift, and nobody can see which combination was live last Tuesday.

### Where our case sits

AcruxCore stores the whole configuration as a **prompt version**. One version holds the messages
*and* the default model together, and it is immutable — commit it and it never changes again.

An **alias** is a named pointer at one of those versions. Two are created for you on the first
commit, `production` and `staging`. But an alias is a **string, not a fixed enum**. Nothing stops
you from creating others:

| Alias | Points at | So a render returns |
|---|---|---|
| `quick` | v1 | the cheap model, the concise persona |
| `deep` | v2 | the stronger model, the thorough persona |
| `production` | v1 | whatever you last promoted |

A name can be anything: `quick`, `deep`, a customer id, a region, a canary.

### The direct answer

Configuration is *which alias you render*. `hub.prompts.render(PROMPT, "quick")` and
`hub.prompts.render(PROMPT, "deep")` are the same call with one different string, and they come
back with different models, different system messages and the same tool. Your code does not
branch, because there is nothing for it to branch on.

### The one exception, and why it exists

Search depth is the exception, and it is worth understanding rather than memorising.

The tool's schema deliberately has **one** argument, `query`. Depth is not in it. So depth cannot
come from the model, and it cannot come from the render either, because it is not part of a
message — it is an argument to a Tavily call that happens inside your process.

That leaves your code, which is why Step 8 builds the `client_tools` map *per alias*: the closure
captures the depth. It is the only `if alias ==` in the whole notebook.

Could you put depth in the schema instead? Yes, and then the **model** would choose it, which is
exactly what you do not want here. The whole design is that depth is a configuration decision, not
a per-question one.

### The trap

Committing a new version does **not** move any alias. Commit v2 and every caller still gets v1,
with no error and nothing in the logs. That is the single most common surprise in this model, and
Step 11 does it on purpose.

---

## Step 2 — Create the `web_research` shell

A tool in the catalog is **two objects**, not one. The **shell** owns the name and the
model-facing description. A **version** owns the argument schema and the executor. This step
creates the shell only.

A shell on its own is not callable. Step 3 makes it callable.

### In the dashboard

**Gateway → Tools → New tool.**

| Field | What to enter |
|---|---|
| **Name** | `web_research` |
| **Description** | `Search the live web for information about a topic.` |

![New tool dialog with the name web research and a description about searching the live web](https://docs.acruxcore.com/img/tutorials/build-a-configurable-react-agent/01-new-tool.png)

The description is the sentence the **model** reads when it decides whether to call this tool.
Write it for the model, not for your teammates.

### The same thing in code

**Setup.** Find-or-create: it looks the name up first, so a second run of this notebook creates
nothing.

In [3]:
TOOL_DESCRIPTION = "Search the live web for information about a topic."


async def find_tool_by_name(name: str):
    """The catalog row with exactly this name, or None.

    A notebook helper, NOT an SDK function. `search=` matches substrings, so the
    exact-name filter has to happen here.
    """
    found = await hub.tools.list(search=name, limit=100)
    return next((t for t in found.data if t.name == name), None)


tool = await find_tool_by_name(TOOL)
if tool is None:
    tool = await hub.tools.create(name=TOOL, description=TOOL_DESCRIPTION)
    print(f"created tool shell {TOOL}")
else:
    print(f"tool {TOOL} already in the catalog")

print("tool id:", tool.id)

tool web_research already in the catalog
tool id: 9de6ab5b-8a35-4ee9-8499-0b90b5500714


---

## Step 3 — Commit version 1, with one argument and no depth

The schema is deliberately thin. **One** argument, `query`. Depth is absent on purpose — Step 1
explains why, and this is the step where that decision is actually made.

The executor is **client**, because your process makes the Tavily call.

### In the dashboard

**Gateway → Tools → `web_research` → New version.**

| Field | What to enter |
|---|---|
| **Parameters** | one row: name `query`, type `string`, **required** |
| **`query` description** | `What to search the web for.` |
| **Executor** | **Client — the caller's app runs it** |

![New version form showing a required query string parameter and the executor set to Client](https://docs.acruxcore.com/img/tutorials/build-a-configurable-react-agent/02-tool-version.png)

Click **Commit version**. The first version automatically gets the `production` and `staging`
aliases. Note that a **tool** has its own aliases, separate from the prompt's.

### The same thing in code

**Setup.** A version is immutable, which is why changing a schema means committing a new version
rather than editing this one.

In [4]:
QUERY_SCHEMA = {
    "type": "object",
    "properties": {
        "query": {"type": "string", "description": "What to search the web for."}
    },
    "required": ["query"],
    # No "depth" here on purpose: depth is configuration, not a model decision.
}

versions = await hub.tools.list_versions(tool.id, limit=1)
if versions.total == 0:
    version = await hub.tools.commit_version(
        tool.id,
        parameters_schema=QUERY_SCHEMA,
        executor={"type": "client"},
        description=TOOL_DESCRIPTION,
    )
    print(f"committed v{version.version_number}, "
          f"aliases now pointing here: {[a.alias for a in version.aliases]}")
else:
    print(f"tool already has {versions.total} version(s) - nothing committed")

tool already has 1 version(s) - nothing committed


---

## Step 4 — Create the prompt shell

The prompt is the thing that will hold two versions. This step creates only its name.

### In the dashboard

**Prompts → New prompt.**

| Field | What to enter |
|---|---|
| **Name** | `web-research-agent` |
| **Description** | `Web research agent whose persona and model change by alias.` |

![New prompt dialog with the name web-research-agent](https://docs.acruxcore.com/img/tutorials/build-a-configurable-react-agent/03-new-prompt.png)

### The same thing in code

**Setup.** Find-or-create on the name only. The versions come next.

In [5]:
async def find_prompt_by_name(name: str):
    """The prompt with exactly this name, or None. A notebook helper, NOT an SDK function."""
    found = await hub.prompts.list(search=name, limit=100)
    return next((p for p in found.data if p.name == name), None)


prompt = await find_prompt_by_name(PROMPT)
if prompt is None:
    prompt = await hub.prompts.create(
        name=PROMPT,
        description="Web research agent whose persona and model change by alias.",
    )
    print(f"created prompt shell {PROMPT}")
else:
    print(f"prompt {PROMPT} already exists")

print("prompt id:", prompt.id)

created prompt shell web-research-agent
prompt id: 065c3824-7619-4d5f-b6a9-85e6129b984a


---

## Step 5 — Commit v1: the quick persona on the cheap model

This version is the whole `quick` configuration in one immutable object: the concise persona, the
cheap model, and a `{{ question }}` variable so the actual question arrives at render time
rather than being baked in.

### In the dashboard

**Prompts → `web-research-agent` → Editor tab.**

| Field | What to enter |
|---|---|
| **Default model** | `gemini-flash` — the cheap, fast one |
| **System message** | the `QUICK_PERSONA` string in the next code cell; it is too long to repeat here, and a second copy would drift |
| **User message** | `{{ question }}` |

![Editor tab with default model gemini-flash and a system message describing the fast shallow-search persona](https://docs.acruxcore.com/img/tutorials/build-a-configurable-react-agent/04-quick-editor.png)

Click **Commit new version**. This is v1, and because it is the first commit `production` and
`staging` both point at it automatically.

### The same thing in code

**Setup.** Committing is not idempotent — call it twice and you get v1 and v2 of the same text. So
this cell checks the version count first, which is what makes the notebook safe to re-run.

In [6]:
QUICK_PERSONA = (
    "You are a fast web-research assistant. Search once with web_research, then answer in two "
    "or three sentences. Prefer the single most relevant source. Do not list every result, do "
    "not add caveats, and do not search twice. Speed matters more than completeness here."
)

DEEP_PERSONA = (
    "You are a thorough web-research analyst. Search with web_research, then synthesise across "
    "several sources rather than summarising one. Group what you found into short labelled "
    "points, name the disagreements between sources where there are any, and say plainly when "
    "the evidence is thin. Completeness matters more than speed here."
)

existing = await hub.prompts.list_versions(prompt.id, limit=100)
committed = {v.version_number for v in existing.data}

if 1 not in committed:
    v1 = await hub.prompts.commit_version(
        prompt.id,
        messages=[
            {"role": "system", "content": QUICK_PERSONA},
            {"role": "user", "content": "{{ question }}"},
        ],
        model=QUICK_MODEL,          # the model is part of the version
    )
    print(f"committed v{v1.version_number} on {v1.model}")
else:
    print("v1 already exists - nothing committed")

committed v1 on gemini-flash


---

## Step 6 — Connect the tool to the prompt

A binding joins the tool to the prompt, so one render returns the messages *and* the tool schema
together.

The important detail for this page: **a binding is not part of a version.** It is a live setting on
the prompt. So you connect the tool once, here, and it stays connected through every version you
commit afterwards — including v2, which does not exist yet.

### In the dashboard

**Prompts → `web-research-agent` → Tools tab → + Connect a tool from the catalog.**

| Field | What to enter |
|---|---|
| **Tool** | `web_research` |
| **Alias** | `production` |
| **Column** | **default** — every alias of the prompt inherits it, including `quick` and `deep` |

![Tools tab with web research connected in the default column](https://docs.acruxcore.com/img/tutorials/build-a-configurable-react-agent/05-tools-tab.png)

It saves straight away; there is no separate commit for a binding.

### The same thing in code

**Setup.** `set_tool_binding` replaces the binding for that tool instead of adding a second one,
so running this twice leaves one row.

In [7]:
binding = await hub.prompts.set_tool_binding(prompt.id, tool.id, tool_alias="production")
print(f"bound {binding.tool_name} @ {binding.tool_alias} -> v{binding.resolved_version_number}")

bindings = await hub.prompts.list_tool_bindings(prompt.id)
print("default tools on this prompt:", [b.tool_name for b in bindings.default])

bound web_research @ production -> v1
default tools on this prompt: ['web_research']


---

## Step 7 — Commit v2: the deep persona on the stronger model

Same prompt, second version. Different persona, different model. The tool binding you just made is
untouched, because bindings are not part of a version.

### In the dashboard

**Prompts → `web-research-agent` → Editor tab**, again.

| Field | What to change |
|---|---|
| **Default model** | `claude-haiku` — the stronger one |
| **System message** | replace it with the `DEEP_PERSONA` string from the Step 5 cell |
| **User message** | leave `{{ question }}` as it is |

![Editor tab now showing default model claude-haiku and a system message describing the thorough deep-search persona](https://docs.acruxcore.com/img/tutorials/build-a-configurable-react-agent/06-deep-editor.png)

Click **Commit new version** to create v2.

### The same thing in code

**Setup.** Watch what this cell prints about the aliases. Committing v2 moved **nothing** — that
is the trap from Step 1, and the next step is the fix.

In [8]:
existing = await hub.prompts.list_versions(prompt.id, limit=100)
committed = {v.version_number for v in existing.data}

if 2 not in committed:
    v2 = await hub.prompts.commit_version(
        prompt.id,
        messages=[
            {"role": "system", "content": DEEP_PERSONA},
            {"role": "user", "content": "{{ question }}"},
        ],
        model=DEEP_MODEL,
    )
    print(f"committed v{v2.version_number} on {v2.model}")
else:
    print("v2 already exists - nothing committed")

print("\naliases right now:", await list_prompt_aliases(prompt.id))
print("note: v2 exists, and no alias points at it yet")

committed v2 on claude-haiku

aliases right now: {'production': 1, 'staging': 1}
note: v2 exists, and no alias points at it yet


---

## Step 8 — Point `quick` at v1 and `deep` at v2

This is the step the whole tutorial is about. Two aliases, two versions.

Promoting a name that does not exist **creates** it. There is no separate "create alias" call —
promote is an upsert. That is why the same endpoint both makes `quick` and later moves it.

### In the dashboard

**Prompts → `web-research-agent` → Versions tab.** Each version row has promote buttons for the
aliases that already exist. For new names, scroll to the **New alias** form at the bottom.

| Field | First alias | Second alias |
|---|---|---|
| **Alias name** | `quick` | `deep` |
| **Version** | **v1** | **v2** |
| then | click **Promote** | click **Promote** |

![Versions tab listing v2 on claude-haiku and v1 on gemini-flash](https://docs.acruxcore.com/img/tutorials/build-a-configurable-react-agent/07-versions-tab.png)

![New alias form with quick typed in the name field and v1 selected in the version dropdown](https://docs.acruxcore.com/img/tutorials/build-a-configurable-react-agent/09-create-alias-form.png)

Afterwards the v1 row carries `PRODUCTION`, `STAGING` and `QUICK` badges, and the v2 row carries
`DEEP` alone. Custom aliases get a **×** to delete them; `production` and `staging` cannot be
deleted.

![Versions tab showing v2 with a deep badge and v1 with production, staging and quick badges](https://docs.acruxcore.com/img/tutorials/build-a-configurable-react-agent/10-alias-badges-on-versions.png)

### The same thing in code

**Setup.** `promote_alias` is an upsert, so this cell both creates the two names and is safe to
re-run.

In [9]:
for alias, version_number in (("quick", 1), ("deep", 2)):
    moved = await hub.prompts.promote_alias(prompt.id, alias, version_number)
    print(f"{moved.alias:>10} -> v{moved.version_number}")

print("\nevery alias on this prompt:", await list_prompt_aliases(prompt.id))

     quick -> v1
      deep -> v2

every alias on this prompt: {'production': 1, 'staging': 1, 'quick': 1, 'deep': 2}


**Check.** Render both aliases and compare. This is the payoff: one call, one different string,
and the model *and* the persona both change.

The dashboard shows the same thing in **Gateway → Playground → Stored prompt**, where picking a
different alias swaps the system message in front of you.

![Playground with web-research-agent selected and the Alias dropdown listing production, staging, quick and deep](https://docs.acruxcore.com/img/tutorials/build-a-configurable-react-agent/11-playground-alias.png)

In [10]:
QUESTION = "What are people saying about small open-source language models?"

for alias in ("quick", "deep"):
    rendered = await hub.prompts.render(PROMPT, alias, {"question": QUESTION})
    system = rendered.messages[0]["content"]
    print(f"alias {alias!r:>8}  -> v{rendered.version_number}  model={rendered.model}")
    print(f"           tools: {[t['function']['name'] for t in rendered.tools]}")
    print(f"           persona opens: {system[:72]}...")
    print()

alias  'quick'  -> v1  model=gemini-flash
           tools: ['web_research']
           persona opens: You are a fast web-research assistant. Search once with web_research, th...

alias   'deep'  -> v2  model=claude-haiku
           tools: ['web_research']
           persona opens: You are a thorough web-research analyst. Search with web_research, then ...



---

## Step 9 — Write the implementation, once per depth

The catalog holds the schema and no body. This cell is the body.

**Your app.** Two things to notice.

First, `search_the_web` takes a `depth` argument that the **model never sees**. It comes from your
configuration, not from the tool call.

Second, `client_tools_for(alias)` is where the alias finally does show up in code. It returns a map
from tool name to a function with the depth already fixed inside it. The loop then calls that
function with only `query`, which is exactly what the schema promised.

This notebook calls Tavily's REST API directly with `requests`. The companion page's Python tab
uses `langchain_community`'s `TavilySearchResults` wrapper instead, which makes this same call —
but that class is deprecated, and its return value drops the `images` array even when you ask for
images. Calling the endpoint directly is one dependency instead of three, and it lets the next cell
show the image difference the wrapper hides.

In [11]:
TAVILY_SEARCH = "https://api.tavily.com/search"

#: The two depth configurations. `quick` also asks for images, which is a visible
#: difference you can count rather than a claim you have to trust.
DEPTHS = {
    "quick": {"search_depth": "basic", "max_results": 5, "include_images": True},
    "deep": {"search_depth": "advanced", "max_results": 10, "include_images": False},
}


def search_the_web(query: str, *, depth: str) -> dict:
    """One real Tavily search. `depth` is configuration - the model never sends it."""
    body = {"query": query, **DEPTHS[depth]}
    res = requests.post(TAVILY_SEARCH, headers={"Authorization": f"Bearer {TAVILY_KEY}"},
                        json=body, timeout=60)
    res.raise_for_status()
    data = res.json()
    return {
        "results": [
            {"title": r["title"], "url": r["url"], "content": r["content"][:400]}
            for r in data.get("results", [])
        ],
        "images": data.get("images", []),
    }


def client_tools_for(alias: str) -> dict:
    """The client_tools map for one alias, with that alias's depth baked into the closure.

    A notebook helper, NOT an SDK function. This is the only place the alias appears in
    code: the model, the persona and the tool schema all arrive from the render instead.
    """
    depth = alias if alias in DEPTHS else "quick"
    return {TOOL: lambda query: search_the_web(query, depth=depth)}


print("depth configurations:", {k: v["search_depth"] for k, v in DEPTHS.items()})

depth configurations: {'quick': 'basic', 'deep': 'advanced'}


**Check.** Prove the two depths really differ before any model is involved. Same query, two
configurations, and the counts should not match.

In [12]:
for alias in ("quick", "deep"):
    out = search_the_web("open-source language models", depth=alias)
    print(f"{alias:>6}: {len(out['results'])} results, {len(out['images'])} images "
          f"({DEPTHS[alias]['search_depth']} depth)")

 quick: 5 results, 5 images (basic depth)
  deep: 10 results, 0 images (advanced depth)


---

## Step 10 — Run the same agent both ways

**Your app.** This is the part that ships, and the thing to notice is what is *not* in it.

There is no model name. No system message. No search depth. `ask(question, alias)` takes the alias
and passes it to two places: `render`, which returns the model and the persona, and
`client_tools_for`, which fixes the depth. Everything else is identical between the two runs.

`trace` names each run and drops both into one session, so they sit side by side afterwards.

In [13]:
async def ask(question: str, alias: str) -> dict:
    """Answer one question with whichever configuration `alias` points at."""
    rendered = await hub.prompts.render(PROMPT, alias, {"question": question})
    result = await hub.gateway.run_prompt_with_tools(
        rendered,
        client_tools=client_tools_for(alias),
        trace={"name": f"{PROMPT}-{alias}", "session_id": "alias-swap-demo"},
    )
    return {
        "alias": alias,
        "model": rendered.model,
        "version": rendered.version_number,
        "answer": result.content,
        "trace_id": result.trace_id,
        "turns": result.iterations,
    }


RUNS = {}
for alias in ("quick", "deep"):
    RUNS[alias] = await ask(QUESTION, alias)
    run = RUNS[alias]
    print(f"=== alias {alias!r}  v{run['version']}  model={run['model']}  "
          f"turns={run['turns']} ===")
    print(run["answer"])
    print(f"\n(trace {run['trace_id']})\n")

=== alias 'quick'  v1  model=gemini-flash  turns=2 ===
Industry sentiment highlights that open-source small language models (SLMs) are becoming practical powerhouses, offering high task-specific accuracy while drastically cutting computing costs and latency. Developers and enterprises increasingly favor them for on-device and edge deployment, data privacy, and fine-tuning specialized workflows without the heavy GPU overhead of massive models. According to analyses like BentoML's guide on open-source SLMs, these models have matured enough to replace frontier LLMs in many production workloads and multi-agent systems.

(trace 111e4a35-1ba6-4bb7-862c-ee982f2eaab0)

=== alias 'deep'  v2  model=claude-haiku  turns=2 ===
Based on my research, here's what people are currently saying about small open-source language models:

## **Performance and Capability**

- **Surprising effectiveness at narrow tasks**: Multiple sources highlight that small models (1B-14B parameters) can match or exceed larg

Read the two answers against each other, not on their own. The `quick` one should be two or three
sentences from one source. The `deep` one should be longer, grouped into points, and drawn from
several. They are answering the identical question.

Three things changed between those two calls, and your code chose none of them: the model came
from the version, the persona came from the version, and the depth came from the alias-keyed
closure. The wording will differ every run; the shape of the difference should not.

---

## Step 11 — Read it back from the API

**Check.** The claim so far is "the alias changed the model". Do not take the answers' word for
it — read the traces and look at which model actually ran.

Two details make this cell longer than you might expect, and both are worth knowing.

**Flush before you read.** Spans are reported in the background so they never slow a request down.
Read a trace the instant a run returns and you can get an incomplete one — the tool span in
particular may still be in the queue. `await hub.gateway.flush()` waits for the queue to drain.

**Spans are a tree, not a list.** `detail.spans` holds the roots only. The `web_research` tool span
is a *child* of the model turn that asked for it, so counting tool calls means walking `.children`
rather than filtering the top level.

Both runs also share one session id, so they are grouped for comparison.

In [14]:
await hub.gateway.flush()      # drain the background span queue before reading


def every_span(spans):
    """Flatten the span tree. A tool span is a child of the model turn that asked for it."""
    for span in spans:
        yield span
        yield from every_span(span.children)


for alias, run in RUNS.items():
    detail = await hub.traces.get(run["trace_id"])
    spans = list(every_span(detail.spans))
    print(f"{alias:>6}: trace {detail.trace.name}   session {detail.trace.session_id}")
    print(f"        spans={detail.trace.span_count}  tokens={detail.trace.total_tokens}  "
          f"cost={detail.trace.total_cost_usd}")
    print(f"        model that really answered: "
          f"{sorted({s.model for s in spans if s.kind == 'llm' and s.model})}")
    print(f"        tools called: {[s.name for s in spans if s.kind == 'tool']}")

 quick: trace web-research-agent-quick   session alias-swap-demo
        spans=3  tokens=1348  cost=None
        model that really answered: ['google/gemini-3.7-flash']
        tools called: ['web_research']
  deep: trace web-research-agent-deep   session alias-swap-demo
        spans=6  tokens=9238  cost=0.01369
        model that really answered: ['claude-haiku-4-5-20251001']
        tools called: ['web_research', 'web_research', 'web_research', 'web_research']


Two things to read in that output.

The model names differ, and they are the **upstream** ids rather than the public names you
registered — `render` gave your code the public name, and the provider answered with its own build
id. That is the proof that the alias reached all the way through.

The costs may not both be filled in. Cost is computed from the prices on the gateway model, so a
model registered without prices reports `None` however much it really cost. If cost matters to you,
fill in the prices when you register the model.

---

## Step 12 — Four ways to get this wrong

Every cell in this step is **broken on purpose**. None of it is app code.

### Mistake 1 — you commit a new version and no alias moves

**Broken on purpose.** The quiet one, and the reason Step 8 exists. Commit v3, change the model in
it, and then render `quick` again. Nothing about `quick` changes, because an alias is a pointer and
committing does not move pointers. No error, no warning.

In [15]:
v3 = await hub.prompts.commit_version(
    prompt.id,
    messages=[
        {"role": "system", "content": "A third persona nobody will ever see."},
        {"role": "user", "content": "{{ question }}"},
    ],
    model=DEEP_MODEL,
)
print(f"committed v{v3.version_number} on {v3.model}")

print("aliases after committing v3:", await list_prompt_aliases(prompt.id))

rendered = await hub.prompts.render(PROMPT, "quick", {"question": QUESTION})
print(f"\nrendering 'quick' still gives v{rendered.version_number} on {rendered.model}")
print("the new version is live for nobody until an alias is promoted to it")

committed v3 on claude-haiku
aliases after committing v3: {'production': 1, 'staging': 1, 'quick': 1, 'deep': 2}

rendering 'quick' still gives v1 on gemini-flash
the new version is live for nobody until an alias is promoted to it


That is the behaviour you want in production — a commit cannot break callers — but it is the thing
that makes people say "I changed the prompt and nothing happened". Committing publishes nothing.
Promoting does.

### Mistake 2 — rendering an alias that does not exist

**Broken on purpose.** An alias is any string, which cuts both ways: a typo is a valid string too.
There is no list of allowed names to validate against, so the error arrives at render time.

In [16]:
from acruxcore.errors import AcruxCoreError

for alias in ("quikc", "production"):
    try:
        rendered = await hub.prompts.render(PROMPT, alias, {"question": QUESTION})
        print(f"{alias!r:>12}: ok  -> v{rendered.version_number} on {rendered.model}")
    except AcruxCoreError as err:
        print(f"{alias!r:>12}: {err.status_code} {err.code} - {err}")

     'quikc': 404 API_ERROR - acruxcore API error 404 for "web-research-agent/quikc"
'production': ok  -> v1 on gemini-flash


### Mistake 3 — promoting to a version that does not exist

**Broken on purpose.** This one is checked at write time rather than deferred, which is the good
kind of failure: you find out when you promote, not when a customer renders.

In [17]:
try:
    await hub.prompts.promote_alias(prompt.id, "deep", 99)
    print("no error - unexpected")
except AcruxCoreError as err:
    print(f"{err.status_code} {err.code}: {err}")

everything = await list_prompt_aliases(prompt.id)
print("\n'deep' is untouched: deep -> v" + str(everything["deep"]))

404 API_ERROR: acruxcore API error 404 promoting prompt alias: Version 99 not found for this prompt

'deep' is untouched: deep -> v2


### Mistake 4 — deleting a built-in alias

**Broken on purpose.** `quick` and `deep` are yours and can be removed. `production` and `staging`
cannot: too much of the platform assumes they exist. The dashboard shows this by giving custom
aliases a **×** and the built-ins none.

There is no SDK method for deleting an alias either, so this cell uses the same `rest` client the
preflight built. That is a normal thing to do — the SDK covers prompts, tools, traces, sessions,
gateway and evaluations, and everything else is a plain HTTP call.

In [18]:
base = f"/prompts/{prompt.id}/aliases"

res = await rest.delete(f"{base}/production")
print(f"deleting 'production': HTTP {res.status_code}")
print("  ", json.dumps(res.json())[:150] if res.content else "(no body)")

# A custom alias, on the other hand, is yours to remove. Recreate it straight after.
res = await rest.delete(f"{base}/quick")
print(f"\ndeleting 'quick':      HTTP {res.status_code}")

moved = await hub.prompts.promote_alias(prompt.id, "quick", 1)
print(f"recreated 'quick' -> v{moved.version_number} (promote is an upsert)")

deleting 'production': HTTP 400
   {"error": {"code": "CANNOT_DELETE_DEFAULT_ALIAS", "message": "Cannot delete the \"production\" alias"}}

deleting 'quick':      HTTP 204
recreated 'quick' -> v1 (promote is an upsert)


---

## Step 13 — Close the client

**Your app.** Traces are reported in the background so they never slow your request down. Closing
the client flushes whatever is still queued. In a script `async with AcruxCore() as hub:` does it
for you; a notebook has no block to leave, so do it by hand.

The raw `rest` client from the preflight needs closing too. It is holding a TCP connection.

In [19]:
await hub.gateway.aclose()
await rest.aclose()
print("flushed")

flushed


---

## What you built

One agent with two personalities, two models and two search depths, and a codebase that knows
about none of them. Changing which configuration is live is now a promote, not a deploy.

### What of this actually ships

This much, and nothing else from this notebook:

```python
import os, requests
from acruxcore import AcruxCore

TAVILY_KEY = os.environ["TAVILY_API_KEY"]
DEPTHS = {
    "quick": {"search_depth": "basic", "max_results": 5, "include_images": True},
    "deep": {"search_depth": "advanced", "max_results": 10, "include_images": False},
}


def search_the_web(query: str, *, depth: str) -> dict:
    res = requests.post("https://api.tavily.com/search",
                        headers={"Authorization": f"Bearer {TAVILY_KEY}"},
                        json={"query": query, **DEPTHS[depth]}, timeout=60)
    res.raise_for_status()
    data = res.json()
    return {"results": [{"title": r["title"], "url": r["url"],
                         "content": r["content"][:400]} for r in data.get("results", [])],
            "images": data.get("images", [])}


async def ask(question: str, alias: str) -> str:
    async with AcruxCore() as hub:
        rendered = await hub.prompts.render("web-research-agent", alias, {"question": question})
        result = await hub.gateway.run_prompt_with_tools(
            rendered,
            client_tools={"web_research": lambda query: search_the_web(query, depth=alias)},
            trace={"name": f"web-research-agent-{alias}", "session_id": "alias-swap-demo"},
        )
        return result.content
```

Note what is absent: no model name, no system message, no `if alias == "deep"` outside the depth
map. Adding a third configuration is a new version plus a promote, and this file does not change.

Everything else was scaffolding:

- `find_tool_by_name` and `find_prompt_by_name` exist so this notebook can be re-run. They are
  notebook helpers, not SDK calls.
- the create-and-commit cells are the dashboard's job, done once.
- every **Check** cell — the preflight, the depth comparison, the two renders, the trace read —
  proves a step worked. None of it belongs in a request path.
- Step 12 is all deliberately broken, and it left a v3 behind that no alias points at.

### What this notebook left in your team

- a tool `web_research` at v1, one `query` argument, `client` executor
- a prompt `web-research-agent` with three versions: v1 quick, v2 deep, v3 the throwaway from
  Mistake 1
- four aliases: `quick` and `production` and `staging` on v1, `deep` on v2
- one binding, inherited by every alias
- a session `alias-swap-demo` holding both runs

### Where to go next

- [Version a prompt](https://docs.acruxcore.com/docs/guides/version-a-prompt) — commits,
  promotes and rollbacks on their own.
- [Connect a tool to a prompt](https://docs.acruxcore.com/docs/guides/connect-a-tool-to-a-prompt)
  — giving one alias a different tool build from the others.
- [Using sessions and traces](https://docs.acruxcore.com/docs/guides/using-sessions-and-traces)
  — group related runs and dig into what happened.